In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 7: the sealed test, ground and air, one look

User go-ahead: Oct 29. Blocks 4 and 8 (sols 1343-1360 and 3069-3110) were not read, scored or plotted before this notebook. **Results are reported only; nothing is refit or tuned afterwards** (user default accepted Oct 29).

- **Ground model, frozen as on the map:** the all-blocks baseline parameters (`fit_params.csv`, "all training blocks") plus `flann_safe_no_ti` refit on all training fitting rows (`map_export.MapResidual`).
- **Air model, frozen as on the map:** `air_hybrid` refit on all training fitting rows with each row's out-of-fold predicted ground (`map_export.air_training_rows`). On test rows its ground input is the frozen ground model's prediction, as on the map.
- **Benchmarks:** the per-(LTST bin, 30-deg Ls bin) mean of REMS ground and air over all training rows, per-bin mean where a cell is empty (`residual.benchmark`).
- **Framing:** both test blocks lie within 1 km of training positions. This is a test in time, not in space.

In [ ]:
# --- config ---
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r
import map_export as m

TEST_BLOCKS = [4, 8]
LS_BIN_DEG = 30
BASE_TOL_K = 1e-6                  # evaluator baseline vs map_export's on training rows
CLAMP_TOL_K = 0.05                 # ground clamp ranges vs Step 5's flann_safe_clamp_ranges.csv
LOBO_MACRO_K = {"ground_model": 4.86, "ground_bench": 5.38, "air_model": 4.052, "air_bench": 4.037}
TABLE = paths.DATA_DIR / "gale_rems_training.csv"
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
CLAMP_CSV = paths.RESIDUAL_OUT_DIR / "flann_safe_clamp_ranges.csv"
AIR_ROWS = paths.AIR_OUT_DIR / "air_rows.csv"
OUT = paths.SEALED_OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)

## Data and frozen models

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(TABLE, allow_test=True))   # the only use of allow_test in the project
is_test = (tbl.split_role == "test").to_numpy()
assert sorted(tbl.block[is_test].unique()) == TEST_BLOCKS, f"test blocks {sorted(tbl.block[is_test].unique())}"
assert not tbl.loc[is_test, "sol"].between(*m.SUSPECT_SOLS).any(), "suspect sols in the test blocks"
print("test sols per block:", tbl[is_test].groupby("block").sol.agg(["min", "max", "nunique"]).to_dict("index"))
print("test rows with ground temperature:", int(is_test.sum()),
      "| with placeholder tau:", int(tbl.tau_is_placeholder[is_test].sum()),
      "| without air temperature:", int(tbl.air_missing[is_test].sum()))
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
tbl["q_abs_wm2"] = 0.0
train = tbl[~is_test].reset_index(drop=True)
test = tbl[is_test].reset_index(drop=True)

fits = pd.read_csv(FIT_PARAMS).set_index("fold")
params = tuple(fits.loc[m.FIT_PARAMS_LABEL, list(t.PARAM_NAMES)].to_numpy(dtype=float))
print("baseline parameters (g_diff, f_ir, ti_scale):", np.round(params, 4))

# baseline on every row, all-blocks parameters
per_sol = ["ls_deg", "tau", "row", "slope_deg", "aspect_deg", "albedo_mean", "ti_themis_tiu"]
assert (tbl.groupby("sol")[per_sol].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
evaluate = t.make_evaluator(sol_inputs, row_sol, t.bin_centre_step(tbl.bin.to_numpy()))
base_all = evaluate(params).copy()
assert np.all(np.isfinite(base_all)), "non-finite baseline"

# ground residual, refit exactly as for the map (training rows only)
fit_frame, fit_target, _, _ = m.residual_training_rows(train, params)
diff = np.abs(fit_frame.baseline_k.to_numpy() - base_all[~is_test][fit_frame.index.to_numpy()]).max()
print(f"max |evaluator baseline - map_export baseline| on training fitting rows = {diff:.2e} K")
assert diff < BASE_TOL_K, "baseline paths differ"
ground_model = m.MapResidual(fit_frame, fit_target)
ref = pd.read_csv(CLAMP_CSV, index_col=0)
for c, (lo, hi) in ground_model.ranges.items():
    tol = CLAMP_TOL_K if c == "baseline_k" else 1e-9
    assert abs(lo - ref.loc["lo", c]) <= tol and abs(hi - ref.loc["hi", c]) <= tol, f"clamp range of {c} differs from Step 5"
print(f"ground residual {ground_model.variant}: {len(fit_frame)} fitting rows; clamp ranges match Step 5")

# air model, refit exactly as for the map
air_frame, air_target = m.air_training_rows(train, pd.read_csv(AIR_ROWS))
air_model = r.FlannModel(r.AIR_VARIANTS[m.AIR_VARIANT], False, air_frame, air_target)
print(f"air model {m.AIR_VARIANT}: {len(air_frame)} fitting rows, {air_model.pipe[-1].coef_.size} basis columns")

## Predictions on the test rows

In [ ]:
test["baseline_k"] = base_all[is_test]
resid, clipped = ground_model.predict(test)
test["ground_pred_k"] = test.baseline_k + resid
print("test rows clipped by the ground clamp ranges (fraction):", {c: round(f, 3) for c, f in clipped.items()})
test["ground_bench_k"], n_fb_g = r.benchmark(train, test, target="ground_temp_k")
test["air_pred_k"] = air_model.predict(test.assign(baseline_k=test.ground_pred_k))[0]
train_air = train[~train.air_missing]
test["air_bench_k"] = np.nan
has_air = ~test.air_missing.to_numpy()
test.loc[has_air, "air_bench_k"], n_fb_a = r.benchmark(train_air, test[has_air], target="air_temp_k")
print(f"benchmark fallback rows: ground {n_fb_g}, air {n_fb_a}")
test["group"] = np.where(test.tau_is_placeholder, "placeholder_tau", "main")
for q, obs in (("ground", "ground_temp_k"), ("air", "air_temp_k")):
    for k in ("pred", "bench"):
        test[f"{q}_{k}_err"] = test[f"{q}_{k}_k"] - test[obs]

## Scores

In [ ]:
def score(g):
    out = {}
    for q in ("ground", "air"):
        gg = g.dropna(subset=[f"{q}_bench_err"])
        out[f"{q}_n"] = len(gg)
        for k in ("pred", "bench"):
            e = gg[f"{q}_{k}_err"]
            out[f"{q}_{k}_rmse_k"] = np.sqrt((e ** 2).mean())
            out[f"{q}_{k}_bias_k"] = e.mean()
    return pd.Series(out)


main = test[test.group == "main"]
by_block = main.groupby("block").apply(score, include_groups=False)
by_group = test.groupby(["block", "group"]).apply(score, include_groups=False)
summary = pd.DataFrame({
    "sealed_macro_main_k": {f"{q}_{k}": by_block[f"{q}_{k}_rmse_k"].mean() for q in ("ground", "air") for k in ("pred", "bench")},
    "sealed_pooled_main_k": {f"{q}_{k}": np.sqrt((main[f"{q}_{k}_err"].dropna() ** 2).mean()) for q in ("ground", "air") for k in ("pred", "bench")},
    "sealed_pooled_all_k": {f"{q}_{k}": np.sqrt((test[f"{q}_{k}_err"].dropna() ** 2).mean()) for q in ("ground", "air") for k in ("pred", "bench")},
    "lobo_macro_k": {"ground_pred": LOBO_MACRO_K["ground_model"], "ground_bench": LOBO_MACRO_K["ground_bench"],
                     "air_pred": LOBO_MACRO_K["air_model"], "air_bench": LOBO_MACRO_K["air_bench"]},
})
pd.options.display.float_format = "{:.2f}".format
print("Main rows (placeholder tau excluded), by block:")
print(by_block.T.to_string())
print()
print("By block and group:")
print(by_group.T.to_string())
print()
print("Summary (pred = model on the map, bench = benchmark):")
print(summary.to_string())

by_hour = main.groupby("ltst_hour").apply(score, include_groups=False)
print()
print("Bias and RMSE by LTST hour (main rows, both blocks):")
print(by_hour[[c for c in by_hour.columns if not c.endswith("_n")]].to_string())

In [ ]:
test.drop(columns=["q_abs_wm2"]).to_csv(OUT / "sealed_rows.csv", index=False)
by_block.to_csv(OUT / "sealed_by_block.csv")
by_group.to_csv(OUT / "sealed_by_group.csv")
summary.to_csv(OUT / "sealed_summary.csv")
by_hour.to_csv(OUT / "sealed_by_ltst_hour.csv")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for i, b in enumerate(TEST_BLOCKS):
    g = main[main.block == b].groupby("ltst_hour")
    for j, (q, obs) in enumerate((("ground", "ground_temp_k"), ("air", "air_temp_k"))):
        h = g[[obs, f"{q}_pred_k", f"{q}_bench_k"]].mean()
        ax = axes[j, i]
        ax.plot(h.index, h[obs], "k.-", label="REMS")
        ax.plot(h.index, h[f"{q}_pred_k"], "m.-", label="model (as on the map)")
        ax.plot(h.index, h[f"{q}_bench_k"], "b.--", label="benchmark")
        s = main[main.block == b].sol
        ax.set_title(f"{q}, block {b}: sols {s.min()}-{s.max()} (sealed test)", fontsize=9)
        ax.set_ylabel("K")
axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("LTST hour")
fig.tight_layout()
fig.savefig(OUT / "sealed_diurnal.png", dpi=120)
print("saved to", OUT)